# RAM Antivirus — Complete End-to-End Master EDA & Data Pipeline
### **Comprehensive Analysis of All 43 Features in NSL-KDD Dataset**
**Objective:** Load raw data, inspect all 43 columns, check data types and statistics, audit null values one-by-one, perform categorical encoding, feature selection, and data scaling in simple, step-by-step flow.

---


In [ ]:
# Optional: Run this if any package is missing in your current kernel
# %pip install -q joblib scikit-learn seaborn matplotlib pandas numpy pyod

import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier

# Configure visual style
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)

print("[OK] Setup completed successfully!")


---
## Step 1: Define All 43 Available Columns
The raw dataset contains 41 operational and host attributes + 1 attack label + 1 difficulty score.


In [ ]:
# All 43 columns present in the raw NSL-KDD dataset
ALL_43_COLUMNS = [
    "duration", "protocol_type", "service", "flag", "src_bytes", "dst_bytes",
    "land", "wrong_fragment", "urgent", "hot", "num_failed_logins", "logged_in",
    "num_compromised", "root_shell", "su_attempted", "num_root", "num_file_creations",
    "num_shells", "num_access_files", "num_outbound_cmds", "is_host_login",
    "is_guest_login", "count", "srv_count", "serror_rate", "srv_serror_rate",
    "rerror_rate", "srv_rerror_rate", "same_srv_rate", "diff_srv_rate",
    "srv_diff_host_rate", "dst_host_count", "dst_host_srv_count",
    "dst_host_same_srv_rate", "dst_host_diff_srv_rate", "dst_host_same_src_port_rate",
    "dst_host_srv_diff_host_rate", "dst_host_serror_rate", "dst_host_srv_serror_rate",
    "dst_host_rerror_rate", "dst_host_srv_rerror_rate", "label", "difficulty"
]

print(f"Total columns defined: {len(ALL_43_COLUMNS)}")


---
## Step 2: Load Raw Dataset & Inspect First / Last Rows


In [ ]:
DATA_DIR = os.path.join("..", "backend", "data")

train_file = os.path.join(DATA_DIR, "KDDTrain+.txt")
test_file = os.path.join(DATA_DIR, "KDDTest+.txt")

df = pd.read_csv(train_file, header=None, names=ALL_43_COLUMNS)
df_test = pd.read_csv(test_file, header=None, names=ALL_43_COLUMNS)

print(f"Dataset successfully loaded!")
print(f"Train Shape: {df.shape[0]:,} Rows × {df.shape[1]} Columns")
print(f"Test Shape:  {df_test.shape[0]:,} Rows × {df_test.shape[1]} Columns")


In [ ]:
# Display first 5 rows (df.head())
df.head()


In [ ]:
# Display last 5 rows (df.tail())
df.tail()


---
## Step 3: Check Data Types & Schema (`df.info()`)


In [ ]:
# Full schema and memory info
df.info()


In [ ]:
# Count of data types
df.dtypes.value_counts()


---
## Step 4: Check for Null / Missing Values (All 43 Columns One by One)
We inspect every single column individually to verify data completeness.


In [ ]:
# Check null values for all 43 columns one by one
null_series = df.isnull().sum()
null_df = pd.DataFrame({
    "Column Name": null_series.index,
    "Missing Values": null_series.values,
    "Missing Percentage (%)": (null_series.values / len(df)) * 100
})

print(f"Total missing values in entire dataset: {null_series.sum()}")
null_df


In [ ]:
# Check for duplicate rows
duplicate_count = df.duplicated().sum()
print(f"Duplicate rows count: {duplicate_count}")


---
## Step 5: Statistical Summary of Numerical Columns (`df.describe()`)
Computes mean, standard deviation, minimum, quartiles, and maximum for all numerical attributes.


In [ ]:
# Statistical summary of numerical columns
df.describe().T


---
## Step 6: Categorical Feature Analysis (`protocol_type`, `service`, `flag`)


In [ ]:
print("=== Protocol Distribution ===")
print(df["protocol_type"].value_counts())

print("\n=== Top 10 Services ===")
print(df["service"].value_counts().head(10))

print("\n=== Top 5 Connection Flags ===")
print(df["flag"].value_counts().head(5))


In [ ]:
# Plotting Categorical Features
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.5))

df["protocol_type"].value_counts().plot(kind="bar", ax=ax1, color="#1971c2")
ax1.set_title("Protocol Type Distribution", fontweight="bold")
ax1.set_ylabel("Count")

df["flag"].value_counts().head(5).plot(kind="bar", ax=ax2, color="#fa5252")
ax2.set_title("Top 5 Connection Flags", fontweight="bold")
ax2.set_ylabel("Count")

plt.tight_layout()
plt.show()


---
## Step 7: Encoding & Target Binarization
1. **Binary Encoding:** We create `is_attack`: $0 = \text{Normal}$, $1 = \text{Attack}$.
2. **Label Encoding:** We convert text columns (`protocol_type`, `service`, `flag`) to numeric values.
3. **Attack Taxonomy:** We map 39 attack types into 4 standard categories (DoS, Probe, R2L, U2R).


In [ ]:
# Target Binarization
df["clean_label"] = df["label"].str.strip().str.lower()
df["is_attack"] = (df["clean_label"] != "normal").astype(int)

# Mapping attack types to families
ATTACK_CATEGORIES = {
    "normal": "Normal",
    "neptune": "DoS", "back": "DoS", "land": "DoS", "pod": "DoS", "smurf": "DoS",
    "teardrop": "DoS", "mailbomb": "DoS", "processtable": "DoS", "udpstorm": "DoS", "apache2": "DoS", "worm": "DoS",
    "ipsweep": "Probe", "nmap": "Probe", "portsweep": "Probe", "satan": "Probe", "mscan": "Probe", "saint": "Probe",
    "ftp_write": "R2L", "guess_passwd": "R2L", "imap": "R2L", "multihop": "R2L", "phf": "R2L", "spy": "R2L",
    "warezclient": "R2L", "warezmaster": "R2L", "sendmail": "R2L", "named": "R2L", "snmpgetattack": "R2L",
    "snmpguess": "R2L", "xlock": "R2L", "xsnoop": "R2L", "httptunnel": "R2L",
    "buffer_overflow": "U2R", "loadmodule": "U2R", "perl": "U2R", "rootkit": "U2R", "ps": "U2R", "xterm": "U2R", "sqlexec": "U2R"
}

df["attack_family"] = df["clean_label"].map(lambda x: ATTACK_CATEGORIES.get(x, "Unknown"))

print("=== Binary Target Counts ===")
print(df["is_attack"].value_counts())
print(f"\nContamination Rate (Attack %): {df['is_attack'].mean()*100:.2f}%")

print("\n=== Attack Family Breakdown ===")
print(df["attack_family"].value_counts())


In [ ]:
# Encoding Categorical Features using LabelEncoder
encoded_df = df.copy()
le_dict = {}

for col in ["protocol_type", "service", "flag"]:
    le = LabelEncoder()
    encoded_df[col] = le.fit_transform(encoded_df[col])
    le_dict[col] = le

print("Categorical columns successfully converted to numerical values!")
encoded_df[["protocol_type", "service", "flag"]].head()


---
## Step 8: Feature Importance & Selection (Random Forest Ranking)
We train a Random Forest on all candidate features to find the most important predictors.  
**Note:** `label` and `difficulty` are excluded to prevent target leakage.


In [ ]:
# Prepare features (exclude target and difficulty)
feature_cols = [c for c in ALL_43_COLUMNS if c not in ["label", "difficulty"]]
X_candidates = encoded_df[feature_cols]
y_target = encoded_df["is_attack"]

# Train Random Forest on sample for speed
sample_idx = np.random.RandomState(42).choice(len(encoded_df), size=15000, replace=False)
rf = RandomForestClassifier(n_estimators=50, random_state=42, n_jobs=-1)
rf.fit(X_candidates.iloc[sample_idx], y_target.iloc[sample_idx])

# Rank features
feature_ranking = pd.Series(rf.feature_importances_, index=feature_cols).sort_values(ascending=False)

print("=== Top 15 Most Important Features ===")
print(feature_ranking.head(15).round(4))


In [ ]:
# Plot Feature Importances
plt.figure(figsize=(10, 6))
feature_ranking.head(15).plot(kind="barh", color="#228be6").invert_yaxis()
plt.title("Top 15 Feature Importances Ranked by Random Forest", fontweight="bold")
plt.xlabel("Importance Score")
plt.tight_layout()
plt.show()


---
## Step 9: Correlation Matrix Heatmap
We select the key operational flow features and plot their pairwise correlations.


In [ ]:
KEY_FLOW_FEATURES = [
    "duration", "src_bytes", "dst_bytes", "count", "srv_count",
    "serror_rate", "rerror_rate", "same_srv_rate", "diff_srv_rate"
]

plt.figure(figsize=(8.5, 6.5))
corr = df[KEY_FLOW_FEATURES].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))

sns.heatmap(corr, mask=mask, cmap="coolwarm", vmin=-1, vmax=1, annot=True, fmt=".2f", square=True)
plt.title("Correlation Matrix of Key Flow Features", fontweight="bold")
plt.tight_layout()
plt.show()


---
## Step 10: Feature Distributions: Normal vs Attack (Boxplots)


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
sample_box = df.sample(n=10000, random_state=42).copy()
sample_box["Class"] = sample_box["is_attack"].map({0: "Normal", 1: "Attack"})

sns.boxplot(data=sample_box, x="Class", y="count", ax=axes[0], palette=["#2b8a3e", "#c92a2a"])
axes[0].set_title("Connection Count (count)", fontweight="bold")

sns.boxplot(data=sample_box, x="Class", y="serror_rate", ax=axes[1], palette=["#2b8a3e", "#c92a2a"])
axes[1].set_title("SYN Error Rate (serror_rate)", fontweight="bold")

sns.boxplot(data=sample_box, x="Class", y="diff_srv_rate", ax=axes[2], palette=["#2b8a3e", "#c92a2a"])
axes[2].set_title("Different Service Rate (diff_srv_rate)", fontweight="bold")

plt.tight_layout()
plt.show()


---
## Step 11: Data Normalization (`StandardScaler`)
We scale numerical flow features to mean 0 and standard deviation 1.


In [ ]:
X_raw = df[KEY_FLOW_FEATURES].fillna(0).values

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_raw)

print(f"Raw shape:    {X_raw.shape}")
print(f"Scaled shape: {X_scaled.shape}")
print(f"Scaled Mean:  {X_scaled.mean(axis=0).round(2)}")
print(f"Scaled Std:   {X_scaled.std(axis=0).round(2)}")
print("\n[OK] General Data Pipeline completed successfully!")
